# VAN ROI y TIR de dos inversiones

Sistemas de Big Data 5074 · UT1.2 · Diapositivas 12–16.

Versión docente resuelta. Contiene la explicación, el código y las salidas. Ejecuta las celdas de arriba abajo en una sesión nueva para reconstruir el proceso.

Solo biblioteca estándar de Python 3.10 o posterior. Los datos están incluidos. No requiere instalaciones ni archivos externos. Puedes subir este `.ipynb` a Colab. Los ejemplos MapReduce se simulan localmente y en serie, sin ejecutar Hadoop.


# Ejercicio 2. Solución y guía docente

## Qué explicar primero

Empiece por distinguir la inversión inicial, los costes anuales y los cobros de cada año. Sitúe el desembolso inicial en el momento cero y los flujos de cada año al final de ese año. Haga una resta a mano antes de abrir las funciones financieras. La comprensión de los signos y de los periodos es el objetivo inicial.

Después explique que el VAN expresa los flujos en el momento inicial usando una tasa de referencia. Presente el ROI con la fórmula elegida para este ejercicio. Finalmente use la TIR como la tasa que hace cero el VAN. El algoritmo numérico ya está preparado para que el alumnado se concentre en su significado económico.

## Trazabilidad con la presentación

| Diapositiva | Contenido original | Tratamiento en este bloque |
|---|---|---|
| 12 | «EJERCICIO 2. Cálculo de indicadores Económicos en un proyecto» | Delimita el ejercicio |
| 13 | «Cuaderno» y logotipo de Google Colab | No contiene un enlace real al cuaderno |
| 14 | Tabla de cuatro años, I0 = 33900, k = 3 % | Caso A |
| 15 | Tabla de cinco años, I0 = 28000, k = 3 %, Ft impresos | Caso B y sensibilidad del Ft del año 5 |
| 16 | VAN, ROI y comparación de TIR con un plazo fijo | Definiciones precisadas e interpretación |

Las tablas son distintas: además de cambiar la inversión y el horizonte, el año 4 pasa de coste 12000 y beneficio 39800 a coste 22000 y beneficio 30000. No son una tabla sin completar y su misma solución.

## Supuestos explícitos y error detectado

- Simulación didáctica en unidades monetarias. La presentación no concreta una moneda.
- «Benefic» se interpreta como cobros antes de restar los costes indicados. Si significase beneficio contable ya neto, volver a restar esos costes duplicaría el descuento de gastos.
- La inversión inicial es un desembolso separado en t=0. Los costes anuales no la incluyen otra vez.
- Todos los periodos son anuales. Los cobros y pagos de cada año ocurren al final del año.
- k = 0,03 es una tasa anual coherente con los importes de la simulación.
- Solo se incluyen los importes aportados. No se añaden valor residual, impuestos, financiación ni otros costes o cobros ausentes de la tabla.
- En la diapositiva 15, 35000 − 32000 = 3000. El Ft impreso de 11000 discrepa en 8000. La solución base conserva las columnas y calcula Ft5 = 3000. La sensibilidad mantiene 11000 como dato alternativo, sin afirmar que concuerde con esas columnas.

## Fórmulas y API

`flujos_anuales(costes, beneficios)` devuelve `[F1, ..., Fn]`, con `Ft = beneficio − coste`.

`construir_flujos(inversion, costes, beneficios)` devuelve `[-I0, F1, ..., Fn]`.

`van(tasa, flujos)` devuelve `sum(CF[t] / (1+tasa)**t)` desde t=0. La tasa se introduce como ratio y debe ser mayor que −1.

`roi(inversion, flujos_anuales)` devuelve `(sum(Ft) − I0) / I0`. Es el ROI del conjunto del proyecto sobre la inversión inicial, sin actualización temporal. El segundo argumento excluye t=0.

`tir(flujos)` devuelve el ratio r que hace cero el VAN. Los flujos deben empezar con un desembolso y tener una única transición de negativo a positivo, ignorando ceros. Puede haber varios desembolsos iniciales y la TIR puede ser negativa. La función resuelve estos flujos convencionales por bisección, en r > −1. No resuelve cualquier secuencia con varios cambios de signo. Los límites de representación de los números de coma flotante también pueden impedir obtener una raíz extrema.

Las funciones devuelven floats sin redondear. Solo la presentación de resultados limita los decimales. La validación rechaza listas vacías, periodos desiguales, importes no finitos e inversiones no positivas. En las tablas, `extraer_columnas` comprueba años consecutivos desde 1.

## Caso A. Diapositiva 14

I0 = 33900. k = 0,03 anual. Duración: cuatro años.

| Año | Coste | Beneficio interpretado como cobro | Ft = beneficio − coste |
|---|---:|---:|---:|
| 1 | 15500 | 3000 | −12500 |
| 2 | 8000 | 20800 | 12800 |
| 3 | 10000 | 26600 | 16600 |
| 4 | 12000 | 39800 | 27800 |

Serie completa: `[-33900, -12500, 12800, 16600, 27800]`.

### VAN manual por periodo

| t | CFt | Operación | Valor en t=0, aproximado |
|---|---:|---|---:|
| 0 | −33900 | −33900 / 1,03⁰ | −33900,000000 |
| 1 | −12500 | −12500 / 1,03¹ | −12135,922330 |
| 2 | 12800 | 12800 / 1,03² | 12065,227637 |
| 3 | 16600 | 16600 / 1,03³ | 15191,351545 |
| 4 | 27800 | 27800 / 1,03⁴ | 24699,939932 |

Sumando sin redondear los términos: **VAN = 5920,596784133568**, que se muestra como **5920,60 unidades monetarias**. Con los supuestos y la tasa del 3 %, el proyecto tiene un excedente actualizado positivo.

### ROI manual

Suma de flujos anuales = −12500 + 12800 + 16600 + 27800 = 44700.

Ganancia neta sin actualizar = 44700 − 33900 = 10800.

`ROI = 10800 / 33900 = 0,3185840707964602`, aproximadamente **31,858407 % para el conjunto del proyecto**.

### TIR e interpretación

A una tasa del 7 %, VAN ≈ 356,804383. Al 8 %, VAN ≈ −888,692066. La tasa que hace cero el VAN está entre ambas.

**TIR ≈ 0,072820976712140, es decir, 7,282097671214 % anual.** Al sustituirla en el VAN, el resultado es prácticamente cero por la precisión numérica. Supera el 3 % de referencia y coincide con la conclusión del VAN positivo en esta serie convencional.

## Caso B. Diapositiva 15 con Ft calculado

I0 = 28000. k = 0,03 anual. Duración: cinco años.

| Año | Coste | Beneficio interpretado como cobro | Ft = beneficio − coste |
|---|---:|---:|---:|
| 1 | 15500 | 3000 | −12500 |
| 2 | 8000 | 20800 | 12800 |
| 3 | 10000 | 26600 | 16600 |
| 4 | 22000 | 30000 | 8000 |
| 5 | 32000 | 35000 | 3000 |

Serie completa: `[-28000, -12500, 12800, 16600, 8000, 3000]`.

### VAN manual por periodo

| t | CFt | Operación | Valor en t=0, aproximado |
|---|---:|---|---:|
| 0 | −28000 | −28000 / 1,03⁰ | −28000,000000 |
| 1 | −12500 | −12500 / 1,03¹ | −12135,922330 |
| 2 | 12800 | 12800 / 1,03² | 12065,227637 |
| 3 | 16600 | 16600 / 1,03³ | 15191,351545 |
| 4 | 8000 | 8000 / 1,03⁴ | 7107,896383 |
| 5 | 3000 | 3000 / 1,03⁵ | 2587,826353 |

Sumando sin redondear los términos: **VAN = −3183,620411444580**, que se muestra como **−3183,62 unidades monetarias**. Los flujos previstos no alcanzan la referencia del 3 %.

### ROI manual

Suma de flujos anuales = −12500 + 12800 + 16600 + 8000 + 3000 = 27900.

Ganancia neta sin actualizar = 27900 − 28000 = −100.

`ROI = −100 / 28000 = −0,0035714285714285713`, aproximadamente **−0,357143 % para el conjunto del proyecto**.

### TIR e interpretación

Al −0,1 %, VAN ≈ 10,150970. Al 0 %, VAN = −100. La raíz está entre esas tasas.

**TIR ≈ −0,000908035517809, es decir, −0,090803551781 % anual.** Una TIR negativa es posible: incluso sin actualizar, los flujos suman menos que la inversión inicial. La conclusión también coincide con el VAN negativo al 3 %.

## Sensibilidad del caso B. Ft5 = 11000 impreso

Se mantiene la misma inversión y se copia la lista de flujos para cambiar exclusivamente el último:

`[-28000, -12500, 12800, 16600, 8000, 11000]`

Este escenario no se obtiene de las columnas de coste y beneficio de la tabla. Para conciliarlo faltaría justificar 8000 adicionales o corregir otro dato. No hay información para elegir una explicación.

El valor actualizado del último flujo pasa a `11000 / 1,03⁵ = 9488,696628225803`. Los demás términos se mantienen.

`VAN alternativo = VAN base + (11000 − 3000) / 1,03⁵`

`= −3183,620411444580 + 6900,870275073312`

**VAN alternativo = 3717,249863628732**, aproximadamente **3717,25 unidades monetarias**.

Suma de flujos anuales = 35900. Ganancia neta = 35900 − 28000 = 7900.

`ROI = 7900 / 28000 = 0,28214285714285714`, aproximadamente **28,214286 % para el conjunto del proyecto**.

Al 6 %, VAN ≈ 93,770908. Al 7 %, VAN ≈ −1005,672842.

**TIR ≈ 0,060835126600598, es decir, 6,083512660060 % anual.**

La errata cambia el signo del VAN y la conclusión al 3 %. El alumnado debe conservar ambas salidas con etiquetas claras y explicar cuál respeta las columnas base.

## Resumen de resultados

| Escenario | VAN al 3 % | ROI del proyecto | TIR anual |
|---|---:|---:|---:|
| Caso A | 5920,60 | 31,858407 % | 7,282098 % |
| Caso B, Ft5 = 3000 | −3183,62 | −0,357143 % | −0,090804 % |
| Sensibilidad, Ft5 = 11000 | 3717,25 | 28,214286 % | 6,083513 % |

Estos casos tienen inversiones y duraciones diferentes. La tabla sirve para resolver los indicadores de cada caso. La presentación no aporta las condiciones necesarias para plantearlos como alternativas mutuamente excluyentes y seleccionar uno únicamente mediante sus porcentajes.

## Otra convención del ROI, solo para el docente

La palabra ROI puede usar denominadores diferentes. Aquí se ha fijado la inversión inicial como denominador. Otra definición sería:

`ROI sobre coste total = (Σbeneficios − Σcostes − I0) / (I0 + Σcostes)`.

En A, los costes anuales suman 45500 y los beneficios 90200. El coste total es 79400 y la ganancia neta es 10800. Esa convención daría `10800 / 79400`, aproximadamente **13,602015 %**.

En B, los costes anuales suman 87500 y los beneficios 115400. El coste total es 115500 y la ganancia neta es −100. Esa convención daría `−100 / 115500`, aproximadamente **−0,086580 %**.

Ambos resultados responden a una definición distinta del denominador. No se deben mezclar. La sensibilidad con Ft5 = 11000 no permite calcular de manera coherente este segundo ROI sin precisar si los 8000 proceden de un mayor beneficio, un menor coste u otro flujo. No añada esta convención al trabajo inicial del alumnado.

## Respuestas a las preguntas del guion

- **¿Coinciden los datos del año 4?** No. A usa coste 12000 y beneficio 39800. B usa coste 22000 y beneficio 30000.
- **¿Qué significa un Ft negativo?** Ese año los costes superan los cobros interpretados. No obliga a que el proyecto completo tenga VAN negativo.
- **¿Por qué hay un elemento adicional en la serie completa?** La inversión inicial ocupa t=0. Los años de explotación empiezan en t=1.
- **¿Qué Ft5 cumple la resta?** 3000. El 11000 impreso difiere en 8000.
- **¿Qué indica el signo del VAN?** Un valor positivo supera la referencia económica fijada por la tasa, uno negativo no la alcanza y cero iguala esa referencia, dentro de los flujos previstos y los supuestos.
- **¿Por qué el ROI no expresa la duración?** Su fórmula suma importes y divide por la inversión sin usar el número de años ni sus fechas. No es una tasa anual y no hay que dividirlo por los años para convertirlo automáticamente en TIR.
- **¿Qué debe dar VAN(TIR)?** Un valor próximo a cero. En coma flotante puede quedar un residuo muy pequeño. No exija igualdad exacta con `== 0`.
- **¿Por qué no equivale la TIR a un plazo fijo garantizado?** La TIR resume una secuencia de flujos prevista, que puede incluir desembolsos posteriores y cobros intermedios. Una igualdad numérica de tasas no iguala el riesgo, la liquidez ni las condiciones del depósito. Reformule la diapositiva 16 como «¿Qué tasa hace que el VAN de estos flujos sea cero?».
- **¿Cambia la conclusión con 11000?** Sí. El caso B pasa de VAN negativo a positivo al 3 %. La discrepancia de datos es material.

## Fallos frecuentes y cómo corregirlos

| Fallo | Corrección |
|---|---|
| Introducir 3 como tasa | Usar 0.03 para representar el 3 % |
| Colocar I0 como positivo | Los desembolsos se introducen con signo negativo |
| Restar I0 dos veces | Incluirlo una sola vez en t=0. El ROI recibe solo flujos anuales |
| Descontar el primer flujo anual en t=0 | Empezar el flujo anual en t=1 |
| Copiar Ft5 = 11000 como si fuese la resta | Calcular 35000 − 32000 y etiquetar por separado la sensibilidad |
| Usar los datos del año 4 de A en B | Mantener cada tabla como caso independiente |
| Mostrar 0.3185 como 0.3185 % | Multiplicar el ratio por 100 al mostrar el porcentaje |
| Interpretar el ROI como anual | Indicar que corresponde al conjunto del proyecto |
| Rechazar toda TIR negativa | Comprobar que la raíz hace cero el VAN y revisar los flujos |
| Redondear cada valor actualizado antes de sumar | Redondear solo para mostrar los resultados |
| Convertir «Benefic» en beneficio neto y restar otra vez el coste | Mantener explícita la interpretación usada |
| Aplicar esta TIR a varios cambios de signo | La función los rechaza porque no es un solucionador general |

## Uso de los archivos

`solucion.py` ejecuta ambos casos y la sensibilidad, imprime un diccionario por escenario y muestra los indicadores con unidades y porcentajes. `resolver_ejercicio()` devuelve el diccionario completo sin imprimir. Los datos constantes permiten copiar el código a Colab sin cargar archivos.

`resolver_caso` llama a `construir_flujos` y utiliza la serie resultante, incluido CF0, para VAN y TIR. Extrae de esa serie los flujos anuales que recibe ROI. La sensibilidad usa una copia de esos flujos anuales y conserva su ruta de cálculo separada.

`alumno.py` prepara importaciones, datos y validaciones. Las funciones principales y el bloque de resolución contienen `raise NotImplementedError` para que el alumnado escriba las restas, bucles y fórmulas. El archivo es sintácticamente válido, pero su ejecución completa se detiene de forma intencionada hasta completar las tareas. El objetivo es calcular VAN y ROI e interpretar la TIR. La validación y la bisección de TIR se proporcionan como apoyo y no se pide al alumnado que las programe.

Los archivos CSV son espejos de las columnas base, sin Ft precalculados. No contienen la sensibilidad ni modifican los datos originales de la presentación.

## Referencias de las fórmulas

La documentación primaria de [numpy_financial.npv](https://numpy.org/numpy-financial/latest/_api_stubs/numpy_financial.npv.html) define la suma desde t=0 con periodos iguales. La de [numpy_financial.irr](https://numpy.org/numpy-financial/latest/_api_stubs/numpy_financial.irr.html) define la TIR como la raíz que hace cero esa suma. Se citan para contrastar las fórmulas. Los archivos de este bloque no usan ni requieren instalar esa biblioteca.


## Código 1 Presentación


In [1]:
"""Ejercicio 2: indicadores económicos de las diapositivas 12–16.

Simulación didáctica. Python 3.10+, solo biblioteca estándar.
Los datos están incluidos para ejecutar el archivo o sus celdas en Colab.
"""


## Código 2 Importaciones


In [2]:
from math import fsum, isfinite
from numbers import Real
from pprint import pprint


## Código 3 Datos de las dos tablas


In [3]:
# Unidades monetarias: la presentación no especifica una moneda.
# Interpretamos «Benefic» como cobros antes de restar los costes indicados.
# La inversión inicial es un desembolso separado en t=0.
# Los cobros y pagos de cada año se sitúan al final de ese año.
TASA_DESCUENTO = 0.03
INVERSION_CASO_A = 33900.0
INVERSION_CASO_B = 28000.0

CASO_A = [
    {"anio": 1, "coste": 15500.0, "beneficio": 3000.0},
    {"anio": 2, "coste": 8000.0, "beneficio": 20800.0},
    {"anio": 3, "coste": 10000.0, "beneficio": 26600.0},
    {"anio": 4, "coste": 12000.0, "beneficio": 39800.0},
]

CASO_B = [
    {"anio": 1, "coste": 15500.0, "beneficio": 3000.0},
    {"anio": 2, "coste": 8000.0, "beneficio": 20800.0},
    {"anio": 3, "coste": 10000.0, "beneficio": 26600.0},
    {"anio": 4, "coste": 22000.0, "beneficio": 30000.0},
    {"anio": 5, "coste": 32000.0, "beneficio": 35000.0},
]


## Código 4 Validación de datos

Este bloque es apoyo ya preparado. No hace falta programar la bisección ni memorizar estas validaciones para resolver los indicadores.


In [4]:
def _numero_finito(valor, nombre):
    if isinstance(valor, bool) or not isinstance(valor, Real):
        raise ValueError(f"{nombre} debe ser un número real.")
    try:
        numero = float(valor)
    except OverflowError as error:
        raise ValueError(f"{nombre} queda fuera del rango numérico.") from error
    if not isfinite(numero):
        raise ValueError(f"{nombre} debe ser finito.")
    return numero


def _serie_finita(valores, nombre):
    try:
        serie = [
            _numero_finito(valor, f"{nombre}[{indice}]")
            for indice, valor in enumerate(valores)
        ]
    except TypeError as error:
        raise ValueError(f"{nombre} debe ser una secuencia de números.") from error
    if not serie:
        raise ValueError(f"{nombre} no puede estar vacío.")
    return serie


def _inversion_positiva(inversion):
    inversion = _numero_finito(inversion, "inversion")
    if inversion <= 0:
        raise ValueError("La inversión inicial debe ser mayor que cero.")
    return inversion


def extraer_columnas(datos):
    """Extrae costes y beneficios de años consecutivos 1, 2, ..., n."""
    if not datos:
        raise ValueError("El caso debe tener al menos un año.")
    costes = []
    beneficios = []
    for anio_esperado, fila in enumerate(datos, start=1):
        anio = fila.get("anio")
        if isinstance(anio, bool) or not isinstance(anio, int) or anio != anio_esperado:
            raise ValueError("Los años deben ser enteros consecutivos desde 1.")
        costes.append(fila["coste"])
        beneficios.append(fila["beneficio"])
    return costes, beneficios


## Código 5 Flujos anuales e inversión inicial


In [5]:
def flujos_anuales(costes, beneficios):
    """Devuelve Ft = beneficio - coste para cada año, sin incluir t=0."""
    costes = _serie_finita(costes, "costes")
    beneficios = _serie_finita(beneficios, "beneficios")
    if len(costes) != len(beneficios):
        raise ValueError("Debe haber un coste y un beneficio por año.")
    if any(valor < 0 for valor in costes + beneficios):
        raise ValueError("En este modelo, costes y beneficios son no negativos.")
    return [beneficio - coste for coste, beneficio in zip(costes, beneficios)]


def construir_flujos(inversion, costes, beneficios):
    """Devuelve [-I0, F1, ..., Fn], con la inversión inicial separada."""
    inversion = _inversion_positiva(inversion)
    return [-inversion] + flujos_anuales(costes, beneficios)


### Situar la inversión en el momento cero

Primero se calculan las restas anuales. La inversión inicial ocupa una posición adicional.


In [6]:
_costes_a, _beneficios_a = extraer_columnas(CASO_A)
_cf_a = construir_flujos(INVERSION_CASO_A, _costes_a, _beneficios_a)
_costes_b, _beneficios_b = extraer_columnas(CASO_B)
_cf_b = construir_flujos(INVERSION_CASO_B, _costes_b, _beneficios_b)
print('Caso A:', _cf_a)
print('Caso B:', _cf_b)


Caso A: [-33900.0, -12500.0, 12800.0, 16600.0, 27800.0]
Caso B: [-28000.0, -12500.0, 12800.0, 16600.0, 8000.0, 3000.0]


## Código 6 Valor actual neto


In [7]:
def van(tasa, flujos):
    """VAN = suma de CF[t] / (1+tasa)**t, empezando en t=0.

    Tasa por periodo como ratio: 0.03 significa 3 %. Los periodos de los
    flujos deben ser iguales a los de la tasa. Devuelve un float sin redondear.
    """
    tasa = _numero_finito(tasa, "tasa")
    if tasa <= -1:
        raise ValueError("La tasa debe ser mayor que -1.")
    flujos = _serie_finita(flujos, "flujos")
    try:
        resultado = fsum(
            flujo / (1 + tasa) ** periodo
            for periodo, flujo in enumerate(flujos)
        )
    except (OverflowError, ZeroDivisionError) as error:
        raise ValueError("Tasa o flujos fuera del rango numérico del cálculo.") from error
    if not isfinite(resultado):
        raise ValueError("El VAN queda fuera del rango numérico del cálculo.")
    return float(resultado)


### Comprobar los términos del VAN

Lee la tabla del caso A por periodos. El primer flujo anual se descuenta una vez y la inversión inicial no se descuenta.


In [8]:
for t, flujo in enumerate(_cf_a):
    actualizado = flujo / (1 + TASA_DESCUENTO)**t
    print(f't={t}, CF={flujo:.0f}, valor en t=0={actualizado:.6f}')
print(f'VAN de A: {van(TASA_DESCUENTO, _cf_a):.6f}')
print(f'VAN de B: {van(TASA_DESCUENTO, _cf_b):.6f}')


t=0, CF=-33900, valor en t=0=-33900.000000
t=1, CF=-12500, valor en t=0=-12135.922330
t=2, CF=12800, valor en t=0=12065.227637
t=3, CF=16600, valor en t=0=15191.351545
t=4, CF=27800, valor en t=0=24699.939932
VAN de A: 5920.596784
VAN de B: -3183.620411


## Código 7 ROI sobre la inversión inicial


In [9]:
def roi(inversion, flujos_anuales):
    """ROI = (suma de Ft - I0) / I0, como ratio y sin actualizar.

    El segundo argumento incluye solo F1, ..., Fn. Multiplicar por 100
    al mostrar el porcentaje. No es una rentabilidad anual.
    """
    inversion = _inversion_positiva(inversion)
    flujos = _serie_finita(flujos_anuales, "flujos_anuales")
    try:
        resultado = (fsum(flujos) - inversion) / inversion
    except OverflowError as error:
        raise ValueError("El ROI queda fuera del rango numérico del cálculo.") from error
    if not isfinite(resultado):
        raise ValueError("El ROI queda fuera del rango numérico del cálculo.")
    return float(resultado)


### Distinguir ratio y porcentaje

Este ROI se refiere al proyecto completo, con I0 como denominador. No es una tasa anual.


In [10]:
for nombre, inversion, serie in [('A', INVERSION_CASO_A, _cf_a), ('B', INVERSION_CASO_B, _cf_b)]:
    valor = roi(inversion, serie[1:])
    print(f'Caso {nombre}: ratio={valor:.9f}, porcentaje={100*valor:.6f} %')


Caso A: ratio=0.318584071, porcentaje=31.858407 %
Caso B: ratio=-0.003571429, porcentaje=-0.357143 %


## Código 8 TIR: función de apoyo numérico ya preparada

Este bloque es apoyo ya preparado. No hace falta programar la bisección ni memorizar estas validaciones para resolver los indicadores.


In [11]:
def tir(flujos):
    """TIR por bisección para flujos convencionales y periodos iguales.

    Exige CF0 < 0 y una única transición de negativo a positivo, ignorando
    ceros. Admite más desembolsos antes del primer cobro neto positivo y
    admite TIR negativa. Busca una raíz r > -1 y devuelve un ratio.

    No es un método general para flujos que alternan varias veces de signo.
    No se pide al alumnado que programe este algoritmo numérico.
    """
    valores = _serie_finita(flujos, "flujos")
    if valores[0] >= 0:
        raise ValueError("La TIR de este modelo requiere un flujo inicial negativo.")
    hay_positivo = False
    for flujo in valores:
        if flujo > 0:
            hay_positivo = True
        elif flujo < 0 and hay_positivo:
            raise ValueError("La TIR admite una sola transición de negativo a positivo.")
    if not hay_positivo:
        raise ValueError("Hace falta al menos un flujo positivo para calcular la TIR.")

    # Los ceros finales no cambian la raíz. La escala evita importes enormes.
    while valores[-1] == 0:
        valores.pop()
    escala = max(abs(valor) for valor in valores)
    valores = [valor / escala for valor in valores]
    ultimo_periodo = len(valores) - 1

    def valor_para_biseccion(tasa):
        base = 1 + tasa
        if base < 1:
            # Es VAN * base**n: conserva su signo para r > -1.
            # En r=-1 solo usamos la extensión del polinomio como frontera.
            return fsum(
                flujo * base ** (ultimo_periodo - periodo)
                for periodo, flujo in enumerate(valores)
            )
        inversa = 1 / base
        return fsum(
            flujo * inversa ** periodo
            for periodo, flujo in enumerate(valores)
        )

    inferior = -1.0
    superior = 1.0
    for _ in range(1024):
        valor_superior = valor_para_biseccion(superior)
        if valor_superior == 0:
            return float(superior)
        if valor_superior < 0:
            break
        superior = superior * 2 + 1
        if not isfinite(superior):
            raise ValueError("No se puede acotar la TIR con precisión numérica.")
    else:
        raise ValueError("No se ha encontrado un intervalo para la TIR.")

    for _ in range(200):
        mitad = inferior + (superior - inferior) / 2
        if mitad == inferior or mitad == superior:
            if mitad <= -1:
                raise ValueError("La TIR está demasiado cerca de -1 para representarla.")
            return float(mitad)
        valor_mitad = valor_para_biseccion(mitad)
        if valor_mitad == 0:
            return float(mitad)
        if valor_mitad > 0:
            inferior = mitad
        else:
            superior = mitad
    raise ValueError("La bisección no ha convergido dentro del límite previsto.")


### Sustituir la TIR en el VAN

La función numérica está dada. La comprobación importante consiste en que el VAN resulte próximo a cero, también cuando la TIR es negativa.


In [12]:
for nombre, serie in [('A', _cf_a), ('B', _cf_b)]:
    tasa = tir(serie)
    print(f'Caso {nombre}: TIR={100*tasa:.9f} %, VAN(TIR)={van(tasa, serie):.9f}')


Caso A: TIR=7.282097671 %, VAN(TIR)=0.000000000
Caso B: TIR=-0.090803552 %, VAN(TIR)=0.000000000


## Código 9 Resolver cada caso y conservar los resultados en diccionarios


In [13]:
def calcular_indicadores(inversion, flujos_del_caso, tasa=TASA_DESCUENTO):
    inversion = _inversion_positiva(inversion)
    anuales = _serie_finita(flujos_del_caso, "flujos_anuales")
    completos = [-inversion] + anuales
    return {
        "inversion_inicial": inversion,
        "tasa_descuento": float(tasa),
        "flujos_anuales": anuales,
        "flujos_desde_t0": completos,
        "van": van(tasa, completos),
        "roi": roi(inversion, anuales),
        "tir": tir(completos),
    }


def resolver_caso(inversion, datos, tasa=TASA_DESCUENTO):
    costes, beneficios = extraer_columnas(datos)
    completos = construir_flujos(inversion, costes, beneficios)
    anuales = completos[1:]
    return {
        "inversion_inicial": float(inversion),
        "tasa_descuento": float(tasa),
        "flujos_anuales": anuales,
        "flujos_desde_t0": completos,
        "van": van(tasa, completos),
        "roi": roi(inversion, anuales),
        "tir": tir(completos),
    }


def resolver_ejercicio():
    caso_a = resolver_caso(INVERSION_CASO_A, CASO_A)
    caso_b = resolver_caso(INVERSION_CASO_B, CASO_B)

    # Sensibilidad: se copia la lista de Ft y se cambia solo el último flujo.
    # No modifica CASO_B ni afirma que 11000 concuerde con 35000 - 32000.
    flujos_sensibilidad = caso_b["flujos_anuales"].copy()
    flujos_sensibilidad[-1] = 11000.0
    sensibilidad = calcular_indicadores(INVERSION_CASO_B, flujos_sensibilidad)
    sensibilidad["aviso"] = (
        "Escenario alternativo con Ft5=11000 impreso. "
        "No concuerda con las columnas base: 35000 - 32000 = 3000."
    )
    return {
        "caso_a_diapositiva_14": caso_a,
        "caso_b_diapositiva_15_corregido": caso_b,
        "sensibilidad_ft5_11000": sensibilidad,
    }


## Código 10 Mostrar resultados


In [14]:
def main():
    resultados = resolver_ejercicio()
    for nombre, resultado in resultados.items():
        print(f"\n{nombre}")
        pprint(resultado, sort_dicts=False)
        print(f"VAN al 3 %: {resultado['van']:.2f} unidades monetarias")
        print(f"ROI del proyecto: {100 * resultado['roi']:.6f} %")
        print(f"TIR anual: {100 * resultado['tir']:.6f} %")
    return resultados


if __name__ == "__main__":
    main()



caso_a_diapositiva_14
{'inversion_inicial': 33900.0,
 'tasa_descuento': 0.03,
 'flujos_anuales': [-12500.0, 12800.0, 16600.0, 27800.0],
 'flujos_desde_t0': [-33900.0, -12500.0, 12800.0, 16600.0, 27800.0],
 'van': 5920.596784133568,
 'roi': 0.3185840707964602,
 'tir': 0.07282097671214027}
VAN al 3 %: 5920.60 unidades monetarias
ROI del proyecto: 31.858407 %
TIR anual: 7.282098 %

caso_b_diapositiva_15_corregido
{'inversion_inicial': 28000.0,
 'tasa_descuento': 0.03,
 'flujos_anuales': [-12500.0, 12800.0, 16600.0, 8000.0, 3000.0],
 'flujos_desde_t0': [-28000.0, -12500.0, 12800.0, 16600.0, 8000.0, 3000.0],
 'van': -3183.62041144458,
 'roi': -0.0035714285714285713,
 'tir': -0.0009080355178092137}
VAN al 3 %: -3183.62 unidades monetarias
ROI del proyecto: -0.357143 %
TIR anual: -0.090804 %

sensibilidad_ft5_11000
{'inversion_inicial': 28000.0,
 'tasa_descuento': 0.03,
 'flujos_anuales': [-12500.0, 12800.0, 16600.0, 8000.0, 11000.0],
 'flujos_desde_t0': [-28000.0, -12500.0, 12800.0, 16600.0